<a href="https://colab.research.google.com/github/SkBlaz/rakun2/blob/main/examples/rakun2_open_source_llm_examples.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RaKUn2 + Open Source LLMs: Keyword Enhancement Examples

This notebook demonstrates how to use **RaKUn2** for fast keyword extraction and then enhance the results using **open-source LLMs** without requiring any external API keys.

We'll show two approaches:
1. **HuggingFace Transformers** - Using models directly from HuggingFace
2. **Ollama** - Using locally running LLMs (when available)

## What is RaKUn2?

RaKUn2 is a very fast keyphrase extraction algorithm suitable for large-scale keyphrase retrieval. It works by:
1. Building a graph of word relationships
2. Using PageRank-like algorithms to score keywords
3. Returning ranked keyword phrases with scores

## Why combine with LLMs?

While RaKUn2 is excellent at identifying important terms, LLMs can help:
- Transform single words into more descriptive phrases
- Add context and domain knowledge
- Improve readability and semantic meaning


## Setup and Installation

In [ ]:
# Install required packages
!pip install rakun2
!pip install transformers torch
!pip install ollama  # Optional, for Ollama examples

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from rakun2 import RakunKeyphraseDetector
import numpy as np
import json

## Sample Text for Analysis

Let's use a sample text about machine learning and AI:

In [ ]:
# Sample document - replace with your own text
sample_text = """
Artificial intelligence and machine learning have revolutionized numerous industries 
through advanced computational techniques. Deep learning, utilizing neural networks 
with multiple hidden layers, enables computers to recognize complex patterns in 
large datasets. Natural language processing allows machines to understand and 
generate human language, while computer vision systems can interpret visual 
information with remarkable accuracy. These technologies power modern applications 
like recommendation systems, autonomous vehicles, fraud detection, and medical 
diagnosis. The combination of big data, cloud computing, and sophisticated 
algorithms has made artificial intelligence more accessible and powerful than 
ever before. Machine learning models continue to improve through techniques 
like transfer learning, reinforcement learning, and ensemble methods.
"""

print("Sample text:")
print(sample_text)

## Step 1: Basic RaKUn2 Keyword Extraction

In [ ]:
# Configure RaKUn2 hyperparameters
hyperparameters = {
    "num_keywords": 10,
    "merge_threshold": 1.1,
    "alpha": 0.3,
    "token_prune_len": 3
}

# Extract keywords
keyword_detector = RakunKeyphraseDetector(hyperparameters)
original_keywords = keyword_detector.find_keywords(sample_text, input_type="string")

print("Original RaKUn2 Keywords:")
print("=" * 50)
for i, (keyword, score) in enumerate(original_keywords, 1):
    print(f"{i:2d}. {keyword:<30} (score: {score:.4f})")

## Step 2: Enhancement with HuggingFace Models

In [ ]:
def enhance_with_huggingface(keywords, model_name="gpt2"):
    """Enhance keywords using HuggingFace transformers"""
    try:
        from transformers import pipeline
        
        # Create text generation pipeline
        print(f"Loading {model_name} model...")
        generator = pipeline(
            "text-generation", 
            model=model_name,
            device=-1,  # Use CPU
            max_length=200,
            do_sample=True,
            temperature=0.7
        )
        
        # Prepare keywords for enhancement
        keywords_str = ", ".join([kw[0] for kw in keywords[:5]])
        
        prompt = f"""Transform these technical keywords into more descriptive phrases:
Keywords: {keywords_str}

Enhanced phrases:
1."""
        
        # Generate enhanced keywords
        response = generator(prompt, max_new_tokens=100, num_return_sequences=1)
        generated_text = response[0]['generated_text']
        
        # Extract enhanced portion
        enhanced_text = generated_text.replace(prompt, "").strip()
        
        # Parse response
        enhanced_keywords = []
        lines = enhanced_text.split('\n')
        
        for i, (original_kw, score) in enumerate(keywords):
            if i < len(lines) and lines[i].strip():
                enhanced = lines[i].strip().lstrip('0123456789. ').strip()
                if enhanced and len(enhanced) > 3:
                    enhanced_keywords.append([enhanced, score])
                else:
                    enhanced_keywords.append([original_kw, score])
            else:
                enhanced_keywords.append([original_kw, score])
        
        return enhanced_keywords
        
    except Exception as e:
        print(f"Error with HuggingFace model: {e}")
        print("Using rule-based enhancement...")
        return rule_based_enhancement(keywords)

def rule_based_enhancement(keywords):
    """Fallback rule-based enhancement"""
    enhancement_rules = {
        'learning': 'machine learning techniques',
        'intelligence': 'artificial intelligence systems',
        'networks': 'neural network architectures', 
        'data': 'big data analytics',
        'algorithms': 'computational algorithms',
        'vision': 'computer vision systems',
        'language': 'natural language processing',
        'systems': 'intelligent systems',
        'detection': 'automated detection methods',
        'models': 'predictive models'
    }
    
    enhanced = []
    for keyword, score in keywords:
        enhanced_kw = keyword
        for key, enhancement in enhancement_rules.items():
            if key in keyword.lower():
                enhanced_kw = enhancement
                break
        enhanced.append([enhanced_kw, score])
    
    return enhanced

In [ ]:
# Enhance keywords with HuggingFace
print("Enhancing keywords with HuggingFace GPT-2...")
print("(This may take a moment on first run)\n")

enhanced_hf_keywords = enhance_with_huggingface(original_keywords)

print("\nHuggingFace Enhanced Keywords:")
print("=" * 50)
for i, (keyword, score) in enumerate(enhanced_hf_keywords, 1):
    print(f"{i:2d}. {keyword:<35} (score: {score:.4f})")

## Step 3: Enhancement with Ollama (Optional)

**Note**: This section requires Ollama to be installed and running locally. In Google Colab, this won't work, but you can run it on your local machine.

To use Ollama:
1. Install Ollama: https://ollama.ai/
2. Pull a model: `ollama pull llama2`
3. Run this code locally

In [ ]:
def enhance_with_ollama(keywords, model_name="llama2"):
    """Enhance keywords using Ollama (for local use)"""
    try:
        import ollama
        
        # Check available models
        available_models = [model['name'] for model in ollama.list()['models']]
        if not available_models:
            print("No Ollama models found. Please install a model first.")
            return rule_based_enhancement(keywords)
        
        if model_name not in available_models:
            model_name = available_models[0]
            print(f"Using available model: {model_name}")
        
        # Prepare prompt
        keywords_list = []
        for i, (keyword, score) in enumerate(keywords[:8], 1):
            keywords_list.append(f"{i}. {keyword}")
        
        keywords_text = "\n".join(keywords_list)
        
        prompt = f"""Improve these technical keywords by making them more descriptive:

{keywords_text}

Transform each into a more descriptive phrase that captures the full meaning. Respond with just the improved phrases, one per line:"""
        
        print(f"Querying {model_name}...")
        response = ollama.chat(model=model_name, messages=[
            {'role': 'user', 'content': prompt}
        ])
        
        response_text = response['message']['content'].strip()
        lines = response_text.split('\n')
        
        enhanced_keywords = []
        for i, (original_kw, score) in enumerate(keywords):
            if i < len(lines) and lines[i].strip():
                enhanced = lines[i].strip().lstrip('0123456789. -*').strip()
                if enhanced and len(enhanced) > 3:
                    enhanced_keywords.append([enhanced, score])
                else:
                    enhanced_keywords.append([original_kw, score])
            else:
                enhanced_keywords.append([original_kw, score])
        
        return enhanced_keywords
        
    except ImportError:
        print("Ollama not available. Install with: pip install ollama")
        return rule_based_enhancement(keywords)
    except Exception as e:
        print(f"Error with Ollama: {e}")
        return rule_based_enhancement(keywords)

In [ ]:
# Try Ollama enhancement (will fallback if not available)
print("Attempting Ollama enhancement...")
enhanced_ollama_keywords = enhance_with_ollama(original_keywords)

print("\nOllama Enhanced Keywords:")
print("=" * 50)
for i, (keyword, score) in enumerate(enhanced_ollama_keywords, 1):
    print(f"{i:2d}. {keyword:<35} (score: {score:.4f})")

## Step 4: Comparison of Results

In [ ]:
print("\n" + "=" * 80)
print("COMPARISON OF KEYWORD ENHANCEMENT METHODS")
print("=" * 80)

print(f"{'Original RaKUn2':<30} | {'HuggingFace Enhanced':<35} | {'Ollama Enhanced'}")
print("-" * 80)

for i in range(min(5, len(original_keywords))):
    orig = original_keywords[i][0]
    hf_enh = enhanced_hf_keywords[i][0] if i < len(enhanced_hf_keywords) else "N/A"
    ollama_enh = enhanced_ollama_keywords[i][0] if i < len(enhanced_ollama_keywords) else "N/A"
    
    print(f"{orig:<30} | {hf_enh:<35} | {ollama_enh}")

print("\n" + "=" * 80)

## Step 5: Try Your Own Text

Now try the workflow with your own text:

In [ ]:
# Enter your own text here
your_text = """
Replace this text with your own document. It could be a research paper abstract,
a blog post, a news article, or any text you want to extract keywords from.
The longer and more technical the text, the better RaKUn2 performs.
"""

# Extract keywords from your text
your_keywords = keyword_detector.find_keywords(your_text, input_type="string")

print("Keywords from your text:")
for i, (keyword, score) in enumerate(your_keywords[:10], 1):
    print(f"{i:2d}. {keyword:<25} (score: {score:.4f})")

# Enhance with rule-based method (always available)
your_enhanced = rule_based_enhancement(your_keywords)
print("\nEnhanced keywords:")
for i, (keyword, score) in enumerate(your_enhanced[:10], 1):
    print(f"{i:2d}. {keyword:<30} (score: {score:.4f})")

## Conclusion

This notebook demonstrated how to:

1. **Extract keywords efficiently** with RaKUn2
2. **Enhance keywords** using open-source LLMs from HuggingFace
3. **Use local LLMs** with Ollama for complete privacy
4. **Compare different enhancement methods**

### Key Benefits:

- **No API keys required** - All models are open-source and free
- **Fast extraction** - RaKUn2 is much faster than LLM-only approaches
- **Better results** - Combining algorithmic extraction with LLM enhancement
- **Privacy-friendly** - Can run completely offline

### Next Steps:

- Try different HuggingFace models (e.g., `microsoft/DialoGPT-medium`, `facebook/blenderbot-400M-distill`)
- Experiment with different RaKUn2 hyperparameters
- Customize the enhancement prompts for your domain
- Use larger Ollama models locally (e.g., `llama2:13b`, `mistral:7b`)

### Resources:

- [RaKUn2 GitHub](https://github.com/SkBlaz/rakun2)
- [HuggingFace Models](https://huggingface.co/models)
- [Ollama](https://ollama.ai/)
